# 从 0 构建 GPT

## 数据准备
读取数据集

In [1]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("length of dataset in characters:", len(text))

length of dataset in characters: 1115394


构建字符表

In [2]:
chars = sorted(list(set(text))) # 列出所有出现的字符列表并排序
vocab_size = len(chars)

print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


简单的 Tokenizer
将字符映射为整数/整数映射为字符

In [3]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # 将字符串转换为整数列表
decode = lambda l: ''.join([itos[i] for i in l]) # 将数字序列转换为字符串

print(encode("my name is moNESY"))
print(decode(encode("my name is moNESY")))

[51, 63, 1, 52, 39, 51, 43, 1, 47, 57, 1, 51, 53, 26, 17, 31, 37]
my name is moNESY


数据集切分

In [4]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) 

print(data.shape, data.dtype)
print(data[:1000])

train_data = data[:n] # 前 90% 用于训练
val_data = data[n:] # 后 10% 用于测试

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

数据加载

In [5]:
torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu' # 有 GPU 跑 GPU
block_size = 8
batch_size = 4

def get_batch(split):
    data = train_data if split == 'train' else val_data 
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) # y 刚好比 x 右移一个字符
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')
print('input:', xb.shape, xb)
print('targets:', yb.shape, yb)
print('-----------')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")


input: torch.Size([4, 8]) tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]], device='cuda:0')
targets: torch.Size([4, 8]) tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]], device='cuda:0')
-----------
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 

## 模型框架
Self-Attention 自注意力机制
| 符号 | 含义           |
| -- | ------------ |
| B  | batch size   |
| T  | 序列长度         |
| C  | embedding 维度 |

x shape = (B, T, C)

head_size = 384 / 6 = 64

In [6]:
import torch.nn as nn
from torch.nn import functional as F
dropout = 0.2 # 随机丢弃率，防止过拟合
n_embd = 384 # 嵌入维度

class Head(nn.Module):
    """ 单个自注意力头 """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # 创建一个下三角矩阵，token_i 只能看 ≤ i 的 token
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)
        # 计算注意力得分 
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # Mask，预测时候不能看未来
        wei = F.softmax(wei, dim=-1) 
        wei = self.dropout(wei)
        # 执行加权聚合
        v = self.value(x) # (B,T,head_size)
        out = wei @ v # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        return out

多头注意力

多个注意力头并行学习不同关系

In [7]:
class MultiHeadAttention(nn.Module):
    """ 多个并行运行的自注意力头 """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)]) # 创建多个 Attention Head
        self.proj = nn.Linear(head_size * num_heads, n_embd) # 把所有 head 拼接后的向量投影回 embedding 维度
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # 输入 x 经过 ModuleList 的每一个头，然后最后一维进行拼接
        out = self.dropout(self.proj(out)) #
        return out

FeedForward

Attention：负责信息交换

FeedForward：负责特征变换

Position-wise Feed Forward Network

In [8]:
class FeedForward(nn.Module):
    """ 简单的线性层加激活函数 """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # FFN(x) = max(0, xW1 + b1)W2 + b2
        return self.net(x)

Transformer Block

整个 GPT 模型其实就是多个 Block 堆叠起来        x
        │
    LayerNorm
        │
 MultiHeadAttention
        │
   Residual Add
        │
    LayerNorm
        │
    FeedForward
        │
   Residual Add
        │
      output

In [9]:
class Block(nn.Module):
    """ Transformer Block: 通信 + 计算 """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

完整框架

In [10]:
n_head = 6 # 多头注意力的头数
n_layer = 6 # Transformer Block 的层数

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) # token id → 向量 (65 , 384)
        self.position_embedding_table = nn.Embedding(block_size, n_embd) # (256 , 384)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # 最后的层归一化
        self.lm_head = nn.Linear(n_embd, vocab_size) # (B,T,vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] # 截断上下文
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] # 提取最后一步
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## 开始训练

In [11]:
@torch.no_grad() # 不要记录梯度
def estimate_loss():
    out = {}
    model.eval() # 测试模式，关闭 dropout
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [12]:
max_iters = 5000 # 训练迭代次数
eval_interval = 500 # 评估间隔
learning_rate = 3e-4 # 学习率
eval_iters = 200

model = GPTLanguageModel()
m = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    optimizer.zero_grad(set_to_none=True)
    logits, loss = model(xb, yb)
    loss.backward()
    optimizer.step()

# 测试生成
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

step 0: train loss 4.1344, val loss 4.1381
step 500: train loss 2.5408, val loss 2.5652
step 1000: train loss 2.4395, val loss 2.4919
step 1500: train loss 2.4569, val loss 2.4625
step 2000: train loss 2.4266, val loss 2.4289
step 2500: train loss 2.4430, val loss 2.4214
step 3000: train loss 2.3967, val loss 2.3842
step 3500: train loss 2.3877, val loss 2.4115
step 4000: train loss 2.3725, val loss 2.3501
step 4500: train loss 2.3493, val loss 2.3619

Baleshets.
Lesche conit,ss ther, chygon thome ash to oee ach: hap.

Hore, thoe to usbl hedin thincoth he utid,
Nooir oarse moye chan des!
Ould
Noua whohe a dey gre;
Whisalit iss arr furfir!
Ow crud whepe doky,
Igh im se toueeficeve aves, thealkeroryooth
ss nod o Ound your ononst wheme mabe fopanle silde thou'llte ryfyaleavee, wamy whikneane hich chy'dlol lexey?
Ceore, wikeespoper ankeanfnot, we wrom beise;
Wheearechs'ch'fs, the beerceagrinnann, feace'dstey,
Nots.
Som pam'd to

I'man shedeaveecy 
